# Step 3: Filter prescriptions to target drugs

### Spark and Hail setup

In [ ]:
import pyspark
import dxpy
import hail as hl

In [ ]:
sc = pyspark.SparkContext()
spark = pyspark.sql.SparkSession(sc)
hl.init(sc=sc, default_reference='GRCh38')

### Environment setup check

In [ ]:
from datetime import datetime
print(f'Timestamp: {datetime.now()}')
print(f'Instance type: {dxpy.describe(dxpy.JOB_ID)["instanceType"]}')
print(f'Hail version: {hl.version()}')
print(f'Spark version: {spark.version}')

### Importing libraries

In [ ]:
import sys
from pprint import pprint

sys.path.append('../')
from prescriptions_processing import DrugsFiltering

### Configuration and Hail tables loading

In [ ]:
input_database = 'prescriptions_db'
input_prescriptions_tb = 'dispensed_prescriptions.ht'

output_database = 'prescriptions_db'
output_filtered_prescriptions_tb = 'filtered_prescriptions_v6.2.0.ht'

preferred_partitioning = 64

In [ ]:
input_db_id = dxpy.find_one_data_object(name=input_database, classname='database', project=dxpy.PROJECT_CONTEXT_ID)['id']
input_prescriptions_ht = hl.read_table(f'dnax://{input_db_id}/{input_prescriptions_tb}')

In [ ]:
input_prescriptions_ht.count()

## Actual prescriptions filtering

In [ ]:
import logging

drugs_filtering = DrugsFiltering(
    input_prescriptions_ht,
    preferred_partitioning=preferred_partitioning,
    logger=DrugsFiltering._default_logger(logging.DEBUG)
)

%time drugs_filtering.run_workflow()

pprint(drugs_filtering.matching_stats)

### Saving results to database

In [ ]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS {output_database} LOCATION 'dnax://'")
output_db_id = dxpy.find_one_data_object(name=output_database, classname='database', project=dxpy.PROJECT_CONTEXT_ID)['id']
output_filtered_prescriptions_url = f'dnax://{output_db_id}/{output_filtered_prescriptions_tb}'

%time drugs_filtering.filtered_prescriptions.write(output_filtered_prescriptions_url, overwrite=True)